<a href="https://colab.research.google.com/github/novay/thesis/blob/main/1.%20base-models/llama32_3b_wikimedia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inisialisasi**

- Llama 3.2-3B-Instruct
- Wikipedia

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

from google.colab import drive
drive.mount('/content/drive')


from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

base_model = "unsloth/Llama-3.2-3B-Instruct"
model_name = "Llama3.2-3B-Instruct"
model_path = "/content/drive/MyDrive/Datasets/Wikipedia/Llama3.2-3B-Instruct/checkpoint-120"

# **1. Preparasi Dataset**

https://huggingface.co/datasets/wikimedia/wikipedia/viewer/20231101.id

In [ ]:
wikipedia_prompt = """Artikel Wikipedia
### Judul: {}

### Artikel:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    titles = examples["title"]
    texts  = examples["text"]
    outputs = []
    for title, text in zip(titles, texts):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = wikipedia_prompt.format(title, text) + EOS_TOKEN
        outputs.append(text)
    return { "text" : outputs, }
pass

### 1.1. Train

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.id", split = "train",)

# We select 10% of the data to make training faster!
dataset = dataset.train_test_split(train_size = 0.1)["train"]
dataset = dataset.map(formatting_prompts_func, batched = True)

dataset.save_to_disk("/content/drive/MyDrive/Datasets/Wikipedia/train_dataset")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/170M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/665622 [00:00<?, ? examples/s]

Map:   0%|          | 0/66562 [00:00<?, ? examples/s]

### 1.2. Evaluasi

In [ ]:
from datasets import load_dataset
from datasets import load_from_disk

eval_dataset = load_dataset("wikimedia/wikipedia", "20231101.id", split="train[1%:2%]")

eval_dataset.save_to_disk("/content/drive/MyDrive/Datasets/Wikipedia/eval_dataset")

### 1.2. Load from Disk

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/Datasets/Wikipedia/train_dataset")
eval_dataset = load_from_disk("/content/drive/MyDrive/Datasets/Wikipedia/eval_dataset")

# **2. Language Fine-tuning**

### 2.1. Inisiasi Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    # token = "hf_...", # Bila menggunakan yang resmi "meta-llama/Llama-2-7b-hf"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head"], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


<a name="Train"></a>
### 2.2. Fine-tuning

In [ ]:
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        # Use warmup_ratio and num_train_epochs for longer runs!
        max_steps = 120,
        warmup_steps = 10,
        # warmup_ratio = 0.1,
        # num_train_epochs = 1,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/Datasets/Wikipedia/Llama3.2-3B-Instruct",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/66562 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66,562 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 982,515,712 of 4,589,267,968 (21.41% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.235900
2,2.019400
3,2.143200
4,2.137100
5,2.016000
6,1.909700
7,2.171700
8,2.081800
9,1.855300
10,2.192300


### 2.3. Log Trainer

In [ ]:
trainer_stats

TrainOutput(global_step=120, training_loss=1.7841061135133107, metrics={'train_runtime': 831.8138, 'train_samples_per_second': 2.308, 'train_steps_per_second': 0.144, 'total_flos': 2.799192370586419e+16, 'train_loss': 1.7841061135133107})

In [ ]:
# Akses log history
log_history = trainer.state.log_history
for log in log_history:
    if "loss" in log:
        print(f"Step {log['step']}: Cross-Entropy Loss = {log['loss']:.4f}")

### 2.4. Save to Disk

In [ ]:
import shutil
import os

source_dir = "./outputs"
destination_dir = "/content/drive/MyDrive/Datasets/Wikipedia/Llama3.2-3B-Instruct/"
os.makedirs(destination_dir, exist_ok=True)
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
print(f"Direktori telah disalin ke {destination_dir}")
print("Isi direktori tujuan:", os.listdir(destination_dir))

Direktori telah disalin ke /content/drive/MyDrive/Datasets/Wikipedia/Llama3.2-3B-Instruct/
Isi direktori tujuan: ['README.md', 'checkpoint-120']


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
6.602 GB of memory reserved.


In [ ]:
from unsloth import FastLanguageModel
import torch

# Konfigurasi
max_seq_length = 2048
dtype = None  # Auto detection (sesuai perangkat)
load_in_4bit = True  # Gunakan kuantisasi 4-bit untuk efisiensi

# Muat model dan tokenizer yang telah difinetune
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/Datasets/Wikipedia/Llama3.2-3B-Instruct/checkpoint-120",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Aktifkan mode inference untuk TTFT rendah
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Unsloth 2025.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): ModulesToSaveWrapper(
          (original_module): Embedding(128256, 3072, padding_idx=128004)
          (modules_to_save): ModuleDict(
            (default): Embedding(128256, 3072, padding_idx=128004)
          )
        )
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=128, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=128, out_features=3072, bias=False)
                )
                (lora_embe

# **3. Evaluasi**

### 3.1. TTFT (Time to First Token)

In [ ]:
# calculate_ttft.py
# Script untuk menghitung TTFT pada model Llama-3.2-3B-Instruct

from unsloth import FastLanguageModel
import torch
import time
import os
import gc

# Set environment variable untuk mengurangi fragmentasi memori
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Bersihkan memori GPU
torch.cuda.empty_cache()
gc.collect()

# Muat model dan tokenizer (lanjutan dari kode Anda)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)
model.to("cuda")  # Pastikan model di GPU

# Format prompt seperti pada finetuning
wikipedia_prompt = """Artikel Wikipedia
### Judul: {}
### Artikel:
{}"""
EOS_TOKEN = tokenizer.eos_token

# Daftar prompt untuk mengukur TTFT
prompts = [
    "Jelaskan secara singkat sejarah Indonesia.",
    "Apa yang dimaksud dengan Pancasila menurut artikel Wikipedia?",
    "Berikan ringkasan tentang budaya Jawa."
]

# Fungsi untuk menghitung TTFT
def calculate_ttft(prompt_text, model, tokenizer, max_seq_length=2048):
    formatted_prompt = wikipedia_prompt.format("Tes", prompt_text) + EOS_TOKEN
    inputs = tokenizer(
        [formatted_prompt],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length
    ).to("cuda")

    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=1)  # Hanya 1 token untuk TTFT
    ttft = time.time() - start_time

    # Bersihkan memori
    torch.cuda.empty_cache()
    return ttft

# Hitung TTFT untuk setiap prompt
results = []
for prompt_text in prompts:
    ttft = calculate_ttft(prompt_text, model, tokenizer)
    results.append({"prompt": prompt_text, "ttft": ttft})

    # Cetak hasil per prompt
    print(f"Prompt: {prompt_text}")
    print(f"TTFT: {ttft:.4f} seconds\n")

# Cetak rata-rata TTFT
avg_ttft = sum([res["ttft"] for res in results]) / len(results)
print(f"Average TTFT: {avg_ttft:.4f} seconds")

==((====))==  Unsloth 2025.7.3: Fast Llama patching. Transformers: 4.53.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Prompt: Jelaskan secara singkat sejarah Indonesia.
TTFT: 2.4966 seconds

Prompt: Apa yang dimaksud dengan Pancasila menurut artikel Wikipedia?
TTFT: 0.1060 seconds

Prompt: Berikan ringkasan tentang budaya Jawa.
TTFT: 0.1038 seconds

Average TTFT: 0.9021 seconds


### 3.2. Perplexity

In [ ]:
%%time

# Format prompt seperti pada finetuning
wikipedia_prompt = """Artikel Wikipedia
### Judul: {}
### Artikel:
{}"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    titles = examples["title"]
    texts = examples["text"]
    outputs = [wikipedia_prompt.format(title, text) + EOS_TOKEN for title, text in zip(titles, texts)]
    return {"text": outputs}


# Set environment variable untuk mengurangi fragmentasi memori
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Bersihkan memori GPU
torch.cuda.empty_cache()
gc.collect()

# Konfigurasi
max_seq_length = 1024  # Dikurangi untuk menghemat memori
dtype = None
load_in_4bit = True
batch_size = 1  # Batch size kecil untuk mencegah out-of-memory


model.to("cuda")  # Pastikan model di GPU

# Terapkan formatting dan hapus semua kolom kecuali 'text'
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True, remove_columns=eval_dataset.column_names)

# Simpan dataset evaluasi untuk konsistensi
eval_dataset.save_to_disk("./eval_wiki_id")

# Tokenisasi dataset evaluasi
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        return_tensors="pt"
    )

tokenized_eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]  # Pastikan hanya input_ids, attention_mask, dll yang tersisa
)
tokenized_eval_dataset.save_to_disk("./tokenized_eval_wiki_id_llama32")

# Buat DataLoader
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
eval_dataloader = DataLoader(tokenized_eval_dataset, batch_size=batch_size, collate_fn=data_collator)

# Hitung Perplexity
model.eval()
total_loss = 0
total_tokens = 0

with torch.no_grad():
    for batch in eval_dataloader:
        inputs = {key: val.to("cuda") for key, val in batch.items() if key in ["input_ids", "attention_mask", "labels"]}
        try:
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss += loss.item() * inputs["input_ids"].size(1)
            total_tokens += inputs["input_ids"].size(1)
        except RuntimeError as e:
            print(f"Error during batch processing: {e}")
            torch.cuda.empty_cache()
            continue
        # Bersihkan memori setelah setiap batch
        torch.cuda.empty_cache()

if total_tokens == 0:
    raise ValueError("No valid tokens processed. Check dataset or model compatibility.")

avg_loss = total_loss / total_tokens
perplexity = torch.exp(torch.tensor(avg_loss)).item()

# Cetak hasil
print(f"Model: {model_name}")
print(f"Perplexity: {perplexity:.2f}")

# Bersihkan memori akhir
torch.cuda.empty_cache()
gc.collect()

Map:   0%|          | 0/6656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6656 [00:00<?, ? examples/s]

Map:   0%|          | 0/6656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6656 [00:00<?, ? examples/s]

Model: Llama-3.2-3B-Instruct
Perplexity: 6.45


95